In [0]:
%sql
/* CREATE VOLUME IF NOT EXISTS workspace.default.lab3_data; */

In [0]:
%sql
/*
CREATE OR REPLACE TABLE customers AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/customers.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE orders AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/orders.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE order_items AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/order_items.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE products AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/products.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE sellers AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/sellers.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE order_payments AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/order_payments.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE order_reviews AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/order_reviews.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE geolocation AS
SELECT * FROM read_files('/Volumes/workspace/default/lab3_data/geolocation.csv', format => 'csv', header => true); */

In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL SELECT 'orders', COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'sellers', COUNT(*) FROM sellers
UNION ALL SELECT 'order_payments', COUNT(*) FROM order_payments
UNION ALL SELECT 'order_reviews', COUNT(*) FROM order_reviews;

Round 2:  Keys and the customer_id Trap

Question 1

In [0]:
%sql
SELECT order_id, COUNT(*) AS row_count
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;

Question 2

In [0]:
%sql
SELECT order_id, COUNT(*) AS row_count
FROM order_items
GROUP BY order_id
HAVING COUNT(*) > 1
LIMIT 10;

SELECT order_id, order_item_id, COUNT(*) AS row_count
FROM order_items
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1;

Question 3

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_id) AS distinct_customer_ids,
  COUNT(DISTINCT customer_unique_id) AS distinct_unique_ids,
  COUNT(DISTINCT customer_name) AS distinct_names 
FROM customers;

Question 4

In [0]:
%sql
SELECT 
  customer_unique_id,
  COUNT(DISTINCT customer_id) AS total_customer_ids
FROM customers
GROUP BY customer_unique_id
ORDER BY total_customer_ids DESC
LIMIT 5;

Round 3: The Double-Counting Trap

Question 1

In [0]:
%sql
SELECT ROUND(SUM(payment_value), 2) AS true_total_payments
FROM order_payments;

Question 2

In [0]:
%sql
SELECT ROUND(SUM(p.payment_value), 2) AS inflated_total_payments
FROM order_items oi
JOIN order_payments p ON oi.order_id = p.order_id;

Question 3

In [0]:
%sql
WITH item_counts AS (
  SELECT order_id, COUNT(*) AS num_items
  FROM order_items
  GROUP BY order_id
  HAVING COUNT(*) = 3
),
payment_counts AS (
  SELECT order_id, COUNT(*) AS num_payments
  FROM order_payments
  GROUP BY order_id
  HAVING COUNT(*) = 2
)
SELECT i.order_id
FROM item_counts i
JOIN payment_counts p ON i.order_id = p.order_id
LIMIT 1;

In [0]:
%sql
SELECT oi.order_id, oi.order_item_id, p.payment_sequential, p.payment_value
FROM order_items oi
JOIN order_payments p ON oi.order_id = p.order_id
WHERE oi.order_id = '9bc94c16-acbd-4e2a-afbb-c3af5dd7c8f8';

Question 4

In [0]:
%sql
WITH items_per_order AS (
  SELECT order_id, SUM(price) AS order_items_total
  FROM order_items
  GROUP BY order_id
),
payments_per_order AS (
  SELECT order_id, SUM(payment_value) AS order_payment_total
  FROM order_payments
  GROUP BY order_id
)
SELECT ROUND(SUM(p.order_payment_total), 2) AS fixed_total_payments
FROM items_per_order i
JOIN payments_per_order p ON i.order_id = p.order_id;

![image_1795486976244.png](./image_1795486976244.png "image_1795486976244.png")